In [2]:
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from hasoc_model import *
%load_ext autoreload
%autoreload 2

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
df_clara = pd.read_csv("hasoc_dataset/train.tsv", sep="\t")
df_clara.columns = ["id", "text", "label_A", "label_B", "label_C"]
df_clara = df_clara[["text", "label_A", "label_B", "label_C"]] 
df_clara = encode_labels(df_clara)

In [4]:
df_claraA = df_clara.dropna(subset=["label_A_enc"])
labelsA = df_claraA["label_A_enc"].tolist()
df_claraA = df_claraA["text"].tolist()

df_claraB = df[df["label_A"] == "HOF"].dropna(subset=["label_B_enc"])
labelsB = df_claraB["label_B_enc"].tolist()
df_claraB = df_claraB["text"].tolist()

df_claraC = df[(df["label_A"] == "HOF") & (df["label_C"].isin(["UNT", "TIN"]))].dropna(subset=["label_C_enc"])
labelsC = df_claraC["label_C_enc"].tolist()
df_claraC = df_claraC["text"].tolist()

In [5]:
#len(df_claraA)
n = 100

df_claraA = df_claraA[0:n]
labelsA = labelsA[0:n]

df_claraB = df_claraB[0:n]
labelsB = labelsB[0:n]

df_claraC = df_claraC[0:n]
labelsC = labelsC[0:n]

In [6]:
MODEL_NAMES = {
    "A": "roberta-base",
    "B": "GroNLP/hateBERT",
    "C": "GroNLP/hateBERT"
}
NUM_LABELS = {"A": 2, "B": 3, "C": 2}

In [7]:
class Paola(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_outputs=8, bin_outputs=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_outputs)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, bin_outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.regressor(pooled), self.classifier(pooled)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_paola = Paola().to(device)
model_paola.load_state_dict(torch.load("model2_loaded.pth", map_location=device, weights_only=True))

print("model2_loaded.pth loaded and ready to use!")

tokenizer_paola = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model2_loaded.pth loaded and ready to use!


In [9]:
class Coline(nn.Module):
    def __init__(self, task, model_name=None, num_labels=None, class_weights=None):
        super().__init__()
        self.task = task
        self.model_name = model_name or MODEL_NAMES[task]
        self.num_labels = num_labels or NUM_LABELS[task]
        self.class_weights = class_weights
        self._keys_to_ignore_on_save = []

        self.transformer = AutoModel.from_pretrained(self.model_name)
        hidden_size = self.transformer.config.hidden_size  # usually 768

        self.extra_feat_size = 13  # 8 numerical + 5 binary

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + self.extra_feat_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, self.num_labels)
        )

        self.loss_fn = nn.CrossEntropyLoss(weight=self.class_weights) if self.class_weights is not None else nn.CrossEntropyLoss()

    def freeze_transformer(self):
        for param in self.transformer.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask, extra_features, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # CLS token

        # Concatenate CLS embedding with extra features
        combined = torch.cat((pooled_output, extra_features), dim=1)

        logits = self.classifier(combined)

        if labels is not None:
            loss = self.loss_fn(logits, labels)

            return {"logits": logits, "loss": loss, "labels": labels}
        return {"logits": logits}


In [10]:
# ------- TASK A ------

In [11]:
task = "A"

In [12]:
encodings_paolaA = tokenizer_paola(df_claraA, truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paolaA = encodings_paolaA['input_ids'].to(device)
attention_mask_paolaA = encodings_paolaA['attention_mask'].to(device)

with torch.no_grad():
    preds_numA, preds_binA = model_paola(input_ids=input_ids_paolaA, attention_mask=attention_mask_paolaA)

preds_numA = preds_numA.cpu().numpy()
preds_binA = preds_binA.cpu().numpy()
preds_binA = (preds_binA > 0.5).astype(int)

for idx, sentence in enumerate(df_claraA[0:5]):
    print(f"Sentence: {sentence}")
    print(f"Numerical predictions: {preds_numA[idx]}")
    print(f"Binary predictions: {preds_binA[idx]}")
    print()

Sentence: #DhoniKeepsTheGlove | WATCH: Sports Minister Kiren Rijiju issues statement backing MS Dhoni over 'Balidaan Badge', tells BCCI to take up the matter with ICC and keep government in the know as nation's pride is involved    https://t.co/zuo5335Rjr
Numerical predictions: [1.4261338  1.026587   0.7156262  0.6578899  1.8180861  0.5279458
 1.3764595  0.03854389]
Binary predictions: [0 0 0 0 1]

Sentence: @politico No. We should remember very clearly that #Individual1 just admitted to treason . #TrumpIsATraitor  #McCainsAHero #JohnMcCainDay
Numerical predictions: [2.8258026  2.305354   1.8951534  1.739517   2.1474311  1.330773
 2.071613   0.32836258]
Binary predictions: [0 0 0 0 0]

Sentence: @cricketworldcup Guess who would be the winner of this #CWC19?     Team who gets maximum points from the abandoned matches 😄 #ShameOnICC #WIvsENG @ICC
Numerical predictions: [2.2002656  1.9434769  1.5539637  1.3744226  2.0864556  0.9715349
 1.9307483  0.07670165]
Binary predictions: [1 0 0 0 0]

numerical_cols = ['sentiment', 'respect', 'insult', 'humiliate', 'status',
                  'dehumanize', 'attack_defend', 'hatespeech']
                  
binary_cols = ['target_race', 'target_religion', 'target_origin', 'target_gender',
               'target_sexuality']

In [13]:
tokenizer_claraA = AutoTokenizer.from_pretrained(MODEL_NAMES[task], use_fast=True)
encodings_claraA = tokenizer_claraA(df_claraA, truncation=True, padding=True, return_tensors="pt")
input_ids_claraA = encodings_claraA['input_ids'].to(device)
attention_mask_claraA = encodings_claraA['attention_mask'].to(device)

In [14]:
class_weightsA = compute_class_weights(labelsA, NUM_LABELS[task], task=task)

model_colineA = Coline(task="A", model_name="roberta-base", class_weights=class_weightsA).to(device)
state_dictA = torch.load("best_model_A_roberta-base.pth", map_location=device, weights_only=True)

# Strip "roberta." from the beginning of keys that belong to the transformer
transformer_state_dictA = {
    k.replace("roberta.", ""): v
    for k, v in state_dictA.items()
    if k.startswith("roberta.")
}

# Load into the RobertaModel (your transformer's structure)
model_colineA.transformer.load_state_dict(transformer_state_dictA, strict=False)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


_IncompatibleKeys(missing_keys=['pooler.dense.weight', 'pooler.dense.bias'], unexpected_keys=[])

In [15]:
extra_featuresA = np.concatenate([preds_numA, preds_binA], axis=1)
extra_features_tensorA = torch.tensor(extra_featuresA, dtype=torch.float32)

In [16]:
datasetA = Dataset.from_dict({
        "input_ids": input_ids_claraA,
        "attention_mask": attention_mask_claraA,
        "labels": torch.tensor(labelsA, dtype=torch.long).tolist(),
        "extra_features": extra_features_tensorA.tolist()
    })

datasetA = datasetA.train_test_split(test_size=0.2, seed=42)

train_model(task, model_colineA, datasetA, tokenizer_claraA, resume=True)

🔁 Resuming from checkpoint: ./results_A_roberta-base/checkpoint-40


/opt/jlab-env/lib/python3.12/site-packages/transformers/trainer.py:3441: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(checkpoint, OPTIMIZER_NAME), m

Epoch,Training Loss,Validation Loss


Task A training complete.


In [17]:
# ------- TASK B ------

In [18]:
task = "B"

In [19]:
encodings_paolaB = tokenizer_paola(df_claraB, truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paolaB = encodings_paolaB['input_ids'].to(device)
attention_mask_paolaB = encodings_paolaB['attention_mask'].to(device)

with torch.no_grad():
    preds_numB, preds_binB = model_paola(input_ids=input_ids_paolaB, attention_mask=attention_mask_paolaB)

preds_numB = preds_numB.cpu().numpy()
preds_binB = preds_binB.cpu().numpy()
preds_binB = (preds_binB > 0.5).astype(int)

for idx, sentence in enumerate(df_claraB[0:5]):
    print(f"Sentence: {sentence}")
    print(f"Numerical predictions: {preds_numB[idx]}")
    print(f"Binary predictions: {preds_binB[idx]}")
    print()

Sentence: @politico No. We should remember very clearly that #Individual1 just admitted to treason . #TrumpIsATraitor  #McCainsAHero #JohnMcCainDay
Numerical predictions: [2.8258026  2.305354   1.8951534  1.739517   2.1474311  1.330773
 2.071613   0.32836258]
Binary predictions: [0 0 0 0 0]

Sentence: #ADOS #trendingnow #blacklivesmatter #justice #fucktrump  ‘Central Park Five’ Prosecutor Elizabeth Lederer Out at Columbia Law School https://t.co/7S1Lu7wi53
Numerical predictions: [2.7250035  2.0836048  1.6560546  1.5017412  2.1553397  1.0952299
 1.8618687  0.20795564]
Binary predictions: [1 0 0 0 0]

Sentence: I don’t know how much more I can take! 45 is a compulsive liar! #Trump30Hours #TrumpIsATraitor
Numerical predictions: [3.6628501 3.5247545 3.2067533 2.8411722 2.9533167 2.1415966 3.05137
 0.7931076]
Binary predictions: [0 0 0 0 0]

Sentence: Good work @ICC keep going just destroy the whole fucking world cup #ShameOnICC https://t.co/ELvq7PAuY9
Numerical predictions: [3.7519     3.5

In [20]:
tokenizer_claraB = AutoTokenizer.from_pretrained(MODEL_NAMES[task], use_fast=True)
encodings_claraB = tokenizer_claraB(df_claraB, truncation=True, padding=True, return_tensors="pt")
input_ids_claraB = encodings_claraB['input_ids'].to(device)
attention_mask_claraB = encodings_claraB['attention_mask'].to(device)

In [21]:
class_weightsB = compute_class_weights(labelsB, NUM_LABELS[task], task=task)

model_colineB = Coline(task="B", model_name="GroNLP/hateBERT", class_weights=class_weightsB).to(device)
state_dictB = torch.load("best_model_B_hateBERT.pth", map_location=device, weights_only=True)

# Adjust the layer names if needed, e.g., by stripping out certain prefixes
model_colineB.transformer.load_state_dict({k.replace("bert.", ""): v for k, v in state_dictB.items()}, strict=False)

_IncompatibleKeys(missing_keys=[], unexpected_keys=['classifier.weight', 'classifier.bias'])

In [22]:
extra_featuresB = np.concatenate([preds_numB, preds_binB], axis=1)
extra_features_tensorB = torch.tensor(extra_featuresB, dtype=torch.float32)

In [23]:
datasetB = Dataset.from_dict({
        "input_ids": input_ids_claraB,
        "attention_mask": attention_mask_claraB,
        "labels": torch.tensor(labelsB, dtype=torch.long).tolist(),
        "extra_features": extra_features_tensorB.tolist()
    })

datasetB = datasetB.train_test_split(test_size=0.2, seed=42)

train_model(task, model_colineB, datasetB, tokenizer_claraB, resume=True)

🔁 Resuming from checkpoint: ./results_B_hateBERT/checkpoint-40


/opt/jlab-env/lib/python3.12/site-packages/transformers/trainer.py:3441: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(checkpoint, OPTIMIZER_NAME), m

Step,Training Loss,Validation Loss


Task B training complete.


In [24]:
# ------- TASK C ------

In [25]:
task = "C"

In [26]:
encodings_paolaC = tokenizer_paola(df_claraB, truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paolaC = encodings_paolaC['input_ids'].to(device)
attention_mask_paolaC = encodings_paolaC['attention_mask'].to(device)

with torch.no_grad():
    preds_numC, preds_binC = model_paola(input_ids=input_ids_paolaC, attention_mask=attention_mask_paolaC)

preds_numC = preds_numC.cpu().numpy()
preds_binC = preds_binC.cpu().numpy()
preds_binC = (preds_binC > 0.5).astype(int)

for idx, sentence in enumerate(df_claraC[0:5]):
    print(f"Sentence: {sentence}")
    print(f"Numerical predictions: {preds_numC[idx]}")
    print(f"Binary predictions: {preds_binC[idx]}")
    print()

Sentence: @politico No. We should remember very clearly that #Individual1 just admitted to treason . #TrumpIsATraitor  #McCainsAHero #JohnMcCainDay
Numerical predictions: [2.8258026  2.305354   1.8951534  1.739517   2.1474311  1.330773
 2.071613   0.32836258]
Binary predictions: [0 0 0 0 0]

Sentence: #ADOS #trendingnow #blacklivesmatter #justice #fucktrump  ‘Central Park Five’ Prosecutor Elizabeth Lederer Out at Columbia Law School https://t.co/7S1Lu7wi53
Numerical predictions: [2.7250035  2.0836048  1.6560546  1.5017412  2.1553397  1.0952299
 1.8618687  0.20795564]
Binary predictions: [1 0 0 0 0]

Sentence: I don’t know how much more I can take! 45 is a compulsive liar! #Trump30Hours #TrumpIsATraitor
Numerical predictions: [3.6628501 3.5247545 3.2067533 2.8411722 2.9533167 2.1415966 3.05137
 0.7931076]
Binary predictions: [0 0 0 0 0]

Sentence: Good work @ICC keep going just destroy the whole fucking world cup #ShameOnICC https://t.co/ELvq7PAuY9
Numerical predictions: [3.7519     3.5

In [27]:
tokenizer_claraC = AutoTokenizer.from_pretrained(MODEL_NAMES[task], use_fast=True)
encodings_claraC = tokenizer_claraC(df_claraC, truncation=True, padding=True, return_tensors="pt")

input_ids_claraC = encodings_claraC['input_ids'].to(device)
attention_mask_claraC = encodings_claraC['attention_mask'].to(device)

In [28]:
class_weightsC = compute_class_weights(labelsC, NUM_LABELS[task], task=task)

model_colineC = Coline(task="C", model_name="GroNLP/hateBERT", class_weights=class_weightsC).to(device)
state_dictC = torch.load("best_model_C_hateBERT.pth", map_location=device, weights_only=True)

model_colineC.transformer.load_state_dict({k.replace("bert.", ""): v for k, v in state_dictC.items()}, strict=False)

_IncompatibleKeys(missing_keys=[], unexpected_keys=['classifier.weight', 'classifier.bias'])

In [29]:
extra_featuresC = np.concatenate([preds_numC, preds_binC], axis=1)
extra_features_tensorC = torch.tensor(extra_featuresC, dtype=torch.float32)

In [30]:
datasetC = Dataset.from_dict({
        "input_ids": input_ids_claraC,
        "attention_mask": attention_mask_claraC,
        "labels": torch.tensor(labelsC, dtype=torch.long).tolist(),
        "extra_features": extra_features_tensorC.tolist()
    })

datasetC = datasetC.train_test_split(test_size=0.2, seed=42)

train_model(task, model_colineC, datasetC, tokenizer_claraC, resume=True)

🔁 Resuming from checkpoint: ./results_C_hateBERT/checkpoint-40


/home/DeepL-Breakers/coline/hasoc_model.py:200: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/jlab-env/lib/python3.12/site-packages/transformers/trainer.py:3441: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any 

Epoch,Training Loss,Validation Loss


Task C training complete.
